# Unlabeled Sample Spectral Analysis

This notebook analyzes unlabeled protein samples (R37S) with general aromatic peaks:
- **Aromatic peak 1**: ~280 nm (Tryptophan/Tyrosine)
- **Aromatic peak 2**: ~310+ nm (secondary aromatic features)

## Key Features
- Optimized for intrinsic protein chromophores
- Tracks peak intensities and positions over time
- Allows peak shifts to track environmental changes
- Analyzes spectral evolution during experiment

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lmfit.models import GaussianModel, LinearModel
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d

%matplotlib inline

## Load Data

In [ ]:
import os
from pathlib import Path

# Create output directories
output_base = Path('spectral_analysis_output')
output_base.mkdir(exist_ok=True)

unlabeled_output = output_base / 'R37S_dose_unlabeled'
unlabeled_output.mkdir(exist_ok=True)

# Load unlabeled data files only (R37S_dose)
datasets = {}

r37s_path = Path('R37S_dose')
for csv_file in sorted(r37s_path.glob('final_pyspec_*.csv')):
    transmission = csv_file.stem.split('_')[-1]  # Extract percentage
    key = f'R37S_{transmission}%'
    data = pd.read_csv(csv_file, index_col=0)
    datasets[key] = {
        'data': data,
        'label': 'Unlabeled (R37S)',
        'transmission': transmission,
        'output_dir': unlabeled_output / f'R37S_{transmission}%'
    }
    datasets[key]['output_dir'].mkdir(exist_ok=True)
    print(f"Loaded {key}: shape {data.shape}")

print(f"\nTotal datasets loaded: {len(datasets)}")
print(f"Output directories created at: {output_base.absolute()}")

## Filter to Relevant Wavelength Range

Focus on 270-450 nm to capture aromatic peaks:

In [ ]:
# Filter to aromatic-relevant wavelengths for all datasets
filtered_datasets = {}

for dataset_name, dataset_info in datasets.items():
    data = dataset_info['data']
    wavelengths = data.index.astype(float)
    data_filtered = data[(wavelengths >= 270) & (wavelengths <= 450)]
    
    filtered_datasets[dataset_name] = {
        **dataset_info,
        'data_filtered': data_filtered
    }
    
    print(f"{dataset_name}:")
    print(f"  Data shape: {data_filtered.shape}")
    print(f"  Wavelength range: {data_filtered.index.astype(float).min():.1f} - {data_filtered.index.astype(float).max():.1f} nm")
    print(f"  Time points: {len(data_filtered.columns)}")

## Peak Detection Helper Function

In [ ]:
def detect_aromatic_peaks(x, y, expected_positions=[280, 310], prominence_factor=0.05):
    """
    Detect aromatic peaks in spectrum.
    
    Parameters:
    -----------
    x : array
        Wavelength values
    y : array
        Absorbance values
    expected_positions : list
        Expected peak wavelengths [280, 310]
    prominence_factor : float
        Minimum prominence as fraction of max absorbance
    
    Returns:
    --------
    dict : Dictionary with peak positions, amplitudes, and widths
    """
    # Smooth the data slightly
    y_smooth = gaussian_filter1d(y, sigma=1)
    
    # Find peaks
    prominence = prominence_factor * (y_smooth.max() - y_smooth.min())
    peaks, properties = find_peaks(y_smooth, prominence=prominence, width=1)
    
    if len(peaks) == 0:
        # If no peaks found, use expected positions
        return {pos: {'center': pos, 'amplitude': 0.1, 'sigma': 10} 
                for pos in expected_positions}
    
    # Match detected peaks to expected positions
    peak_info = {}
    for expected_pos in expected_positions:
        peak_wavelengths = x[peaks]
        distances = np.abs(peak_wavelengths - expected_pos)
        
        if len(distances) > 0:
            closest_idx = np.argmin(distances)
            if distances[closest_idx] < 40:  # Within 40 nm
                peak_idx = peaks[closest_idx]
                peak_info[expected_pos] = {
                    'center': x[peak_idx],
                    'amplitude': y[peak_idx],
                    'sigma': properties['widths'][closest_idx] * (x[1] - x[0])
                }
            else:
                # Use expected position
                peak_info[expected_pos] = {
                    'center': expected_pos,
                    'amplitude': np.interp(expected_pos, x, y),
                    'sigma': 10
                }
        else:
            peak_info[expected_pos] = {
                'center': expected_pos,
                'amplitude': 0.1,
                'sigma': 10
            }
    
    return peak_info

## Define Fitting Model for Unlabeled Sample

Model components:
- **Peak 1**: Aromatic at ~280 nm - center allowed to vary ±20 nm
- **Peak 2**: Aromatic at ~310 nm - center allowed to vary ±20 nm
- **Baseline**: Linear baseline

In [ ]:
def create_aromatic_model(x, y, previous_fit=None):
    """
    Create model for unlabeled sample with two aromatic peaks.
    ALWAYS detects peaks on current spectrum for adaptive fitting.
    
    Parameters:
    -----------
    x : array
        Wavelength values
    y : array
        Absorbance values
    previous_fit : lmfit.ModelResult or None
        Previous fit result (used for reference but not constraints)
    
    Returns:
    --------
    model : lmfit Model
        Combined model
    params : lmfit.Parameters
        Initial parameters
    """
    # ALWAYS detect peaks on the current spectrum
    # This allows tracking of peak shifts and evolution over time
    peak_info = detect_aromatic_peaks(x, y, expected_positions=[280, 310])
    
    # Peak 1: Aromatic at ~280 nm
    peak1 = GaussianModel(prefix='aromatic280_')
    info = peak_info[280]
    params1 = peak1.make_params(
        amplitude=dict(value=info['amplitude'], min=0),
        center=dict(value=info['center'], min=260, max=300),  # Allow ±20 nm shift
        sigma=dict(value=max(5, info['sigma']), min=3, max=25)
    )
    
    # Peak 2: Aromatic at ~310 nm
    peak2 = GaussianModel(prefix='aromatic310_')
    info = peak_info[310]
    params2 = peak2.make_params(
        amplitude=dict(value=info['amplitude'], min=0),
        center=dict(value=info['center'], min=290, max=330),  # Allow ±20 nm shift
        sigma=dict(value=max(5, info['sigma']), min=3, max=25)
    )
    
    # Linear baseline
    baseline = LinearModel(prefix='baseline_')
    params_base = baseline.make_params(
        slope=dict(value=0, min=-0.01, max=0.01),
        intercept=dict(value=np.median(y), min=-1, max=2)
    )
    
    # Combine model and parameters
    model = peak1 + peak2 + baseline
    params = params1 + params2 + params_base
    
    return model, params

## Fit All Time Points

In [ ]:
# Storage for fit results
all_fit_results = {}

for dataset_name, dataset_info in filtered_datasets.items():
    print(f"\n{'='*60}")
    print(f"Processing: {dataset_name}")
    print(f"{'='*60}")
    
    data_filtered = dataset_info['data_filtered']
    fit_results = []
    previous_fit = None
    
    # Fit each time point
    for i, time_point in enumerate(data_filtered.columns):
        x = data_filtered.index.values.astype(float)
        y = data_filtered[time_point].values
        
        # Create model
        model, params = create_aromatic_model(x, y, previous_fit)
        
        # Fit
        try:
            result = model.fit(y, params, x=x, method='leastsq', max_nfev=10000)
            fit_results.append(result)
            previous_fit = result
        except Exception as e:
            print(f"Warning: Fit failed for time point {time_point}: {e}")
            fit_results.append(None)
        
        if (i + 1) % 10 == 0:
            print(f"  Fitted {i + 1}/{len(data_filtered.columns)} time points")
    
    all_fit_results[dataset_name] = fit_results
    print(f"Completed fitting {len([f for f in fit_results if f is not None])}/{len(data_filtered.columns)} time points")

## Visualize First Fit

In [ ]:
# Visualize first fit for each dataset and save
for dataset_name, fit_results in all_fit_results.items():
    dataset_info = filtered_datasets[dataset_name]
    output_dir = dataset_info['output_dir']
    data_filtered = dataset_info['data_filtered']
    
    if fit_results[0] is not None:
        result = fit_results[0]
        time_point = data_filtered.columns[0]
        x = data_filtered.index.values.astype(float)
        y = data_filtered[time_point].values
        comps = result.eval_components(x=x)
        
        plt.figure(figsize=(12, 7))
        plt.scatter(x, y, label=f'Data ({time_point})', s=10, alpha=0.6, color='black')
        plt.plot(x, result.best_fit, label='Best Fit', color='red', linewidth=2.5)
        plt.plot(x, comps['aromatic280_'], label='Aromatic 280nm', linestyle='--', linewidth=2, color='#ff7f0e')
        plt.plot(x, comps['aromatic310_'], label='Aromatic 310nm', linestyle='--', linewidth=2, color='#2ca02c')
        plt.plot(x, comps['baseline_'], label='Baseline', linestyle=':', linewidth=1.5, color='gray')
        
        # Mark peak positions
        aromatic280_center = result.params['aromatic280_center'].value
        aromatic310_center = result.params['aromatic310_center'].value
        plt.axvline(aromatic280_center, color='#ff7f0e', linestyle=':', alpha=0.5)
        plt.axvline(aromatic310_center, color='#2ca02c', linestyle=':', alpha=0.5)
        
        plt.xlabel('Wavelength (nm)', fontsize=12)
        plt.ylabel('Absorbance', fontsize=12)
        plt.title(f'{dataset_name} - First Timepoint\n280nm: {aromatic280_center:.1f} nm, 310nm: {aromatic310_center:.1f} nm', 
                  fontsize=14)
        plt.legend(fontsize=10)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        # Save plot
        output_file = output_dir / f'01_first_fit.png'
        plt.savefig(output_file, dpi=150, bbox_inches='tight')
        plt.close()
        
        print(f"{dataset_name} - First fit saved to {output_file}")
        print(f"  Reduced χ²: {result.redchi:.6f}")
        print(f"  R²: {1 - result.residual.var() / np.var(y):.6f}")

## Extract Peak Parameters Over Time

In [ ]:
# Extract parameters for all datasets
all_parameters = {}

for dataset_name, fit_results in all_fit_results.items():
    dataset_info = filtered_datasets[dataset_name]
    data_filtered = dataset_info['data_filtered']
    
    times = data_filtered.columns.to_list()
    valid_fits = [f for f in fit_results if f is not None]
    valid_indices = [i for i, f in enumerate(fit_results) if f is not None]
    valid_times = [times[i] for i in valid_indices]
    
    if len(valid_fits) > 0:
        aromatic280_centers = [fit.params['aromatic280_center'].value for fit in valid_fits]
        aromatic280_amps = [fit.params['aromatic280_amplitude'].value for fit in valid_fits]
        aromatic280_widths = [fit.params['aromatic280_sigma'].value for fit in valid_fits]
        
        aromatic310_centers = [fit.params['aromatic310_center'].value for fit in valid_fits]
        aromatic310_amps = [fit.params['aromatic310_amplitude'].value for fit in valid_fits]
        aromatic310_widths = [fit.params['aromatic310_sigma'].value for fit in valid_fits]
        
        # Convert times to numeric
        try:
            times_numeric = [float(str(t).replace('s', '')) if isinstance(t, str) else float(t) for t in valid_times]
            time_label = 'Time (s)'
        except (ValueError, TypeError, AttributeError):
            times_numeric = list(range(len(valid_times)))
            time_label = 'Time Point'
        
        all_parameters[dataset_name] = {
            'times': valid_times,
            'times_numeric': times_numeric,
            'time_label': time_label,
            'aromatic280_centers': aromatic280_centers,
            'aromatic280_amps': aromatic280_amps,
            'aromatic280_widths': aromatic280_widths,
            'aromatic310_centers': aromatic310_centers,
            'aromatic310_amps': aromatic310_amps,
            'aromatic310_widths': aromatic310_widths,
        }

print(f"Extracted parameters from {len(all_parameters)} datasets")

## Analyze Peak Positions Over Time

In [ ]:
# Save peak position plots for each dataset
for dataset_name, params in all_parameters.items():
    output_dir = filtered_datasets[dataset_name]['output_dir']
    times_numeric = params['times_numeric']
    time_label = params['time_label']
    aromatic280_centers = params['aromatic280_centers']
    aromatic280_widths = params['aromatic280_widths']
    aromatic310_centers = params['aromatic310_centers']
    aromatic310_widths = params['aromatic310_widths']
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
    
    # Aromatic 280 peak position
    ax1.plot(times_numeric, aromatic280_centers, 'o-', color='#ff7f0e', linewidth=2, markersize=6)
    ax1.axhline(280, color='red', linestyle='--', alpha=0.5, label='Expected (280 nm)')
    ax1.fill_between(times_numeric,
                      [c - w for c, w in zip(aromatic280_centers, aromatic280_widths)],
                      [c + w for c, w in zip(aromatic280_centers, aromatic280_widths)],
                      alpha=0.2, color='#ff7f0e', label='±1σ width')
    ax1.set_ylabel('Wavelength (nm)', fontsize=12)
    ax1.set_title(f'{dataset_name} - Aromatic 280nm Peak Position Over Time', fontsize=13, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Aromatic 310 peak position
    ax2.plot(times_numeric, aromatic310_centers, 's-', color='#2ca02c', linewidth=2, markersize=6)
    ax2.axhline(310, color='red', linestyle='--', alpha=0.5, label='Expected (310 nm)')
    ax2.fill_between(times_numeric,
                      [c - w for c, w in zip(aromatic310_centers, aromatic310_widths)],
                      [c + w for c, w in zip(aromatic310_centers, aromatic310_widths)],
                      alpha=0.2, color='#2ca02c', label='±1σ width')
    ax2.set_xlabel(time_label, fontsize=12)
    ax2.set_ylabel('Wavelength (nm)', fontsize=12)
    ax2.set_title(f'{dataset_name} - Aromatic 310nm Peak Position Over Time', fontsize=13, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save plot
    output_file = output_dir / f'02_peak_positions.png'
    plt.savefig(output_file, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"{dataset_name}")
    print(f"  Aromatic 280: {np.min(aromatic280_centers):.2f} - {np.max(aromatic280_centers):.2f} nm (Δ = {np.max(aromatic280_centers) - np.min(aromatic280_centers):.2f} nm)")
    print(f"  Aromatic 310: {np.min(aromatic310_centers):.2f} - {np.max(aromatic310_centers):.2f} nm (Δ = {np.max(aromatic310_centers) - np.min(aromatic310_centers):.2f} nm)")
    print(f"  Saved to {output_file}")

## Analyze Peak Amplitudes (Aromatic Content)

In [ ]:
# Save peak amplitude plots for each dataset
for dataset_name, params in all_parameters.items():
    output_dir = filtered_datasets[dataset_name]['output_dir']
    times_numeric = params['times_numeric']
    time_label = params['time_label']
    aromatic280_amps = params['aromatic280_amps']
    aromatic310_amps = params['aromatic310_amps']
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Aromatic 280 amplitude
    axes[0].plot(times_numeric, aromatic280_amps, 'o-', color='#ff7f0e', linewidth=2, markersize=6)
    axes[0].set_xlabel(time_label, fontsize=12)
    axes[0].set_ylabel('Amplitude', fontsize=12)
    axes[0].set_title(f'{dataset_name} - Aromatic 280nm Peak Amplitude', fontsize=12, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    
    # Aromatic 310 amplitude
    axes[1].plot(times_numeric, aromatic310_amps, 's-', color='#2ca02c', linewidth=2, markersize=6)
    axes[1].set_xlabel(time_label, fontsize=12)
    axes[1].set_ylabel('Amplitude', fontsize=12)
    axes[1].set_title(f'{dataset_name} - Aromatic 310nm Peak Amplitude', fontsize=12, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save plot
    output_file = output_dir / f'03_peak_amplitudes.png'
    plt.savefig(output_file, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"{dataset_name}")
    print(f"  Aromatic 280 amplitude range: {np.min(aromatic280_amps):.4f} - {np.max(aromatic280_amps):.4f}")
    print(f"  Aromatic 310 amplitude range: {np.min(aromatic310_amps):.4f} - {np.max(aromatic310_amps):.4f}")
    print(f"  Saved to {output_file}")

## Plot Multiple Time Points

In [ ]:
# Plot multiple time points for each dataset and save
for dataset_name, fit_results in all_fit_results.items():
    dataset_info = filtered_datasets[dataset_name]
    output_dir = dataset_info['output_dir']
    data_filtered = dataset_info['data_filtered']
    
    # Plot every Nth spectrum
    n_plots = min(6, len([f for f in fit_results if f is not None]))
    valid_indices = [i for i, f in enumerate(fit_results) if f is not None]
    step = max(1, len(valid_indices) // n_plots)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, i in enumerate(valid_indices[::step][:n_plots]):
        time_point = data_filtered.columns[i]
        result = fit_results[i]
        x = data_filtered.index.values.astype(float)
        y = data_filtered[time_point].values
        
        axes[idx].scatter(x, y, s=5, alpha=0.6, color='black')
        axes[idx].plot(x, result.best_fit, 'r-', linewidth=2)
        axes[idx].set_title(f'Time: {time_point}', fontsize=10)
        axes[idx].set_xlabel('Wavelength (nm)', fontsize=9)
        axes[idx].set_ylabel('Absorbance', fontsize=9)
        axes[idx].grid(True, alpha=0.3)
    
    # Hide unused axes
    for idx in range(len(valid_indices[::step][:n_plots]), n_plots):
        axes[idx].axis('off')
    
    plt.suptitle(f'{dataset_name} - Multiple Time Points', fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    # Save plot
    output_file = output_dir / f'04_multiple_timepoints.png'
    plt.savefig(output_file, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"{dataset_name} - Saved {n_plots} time points to {output_file}")

## Save Results to CSV

In [ ]:
# Create results dataframe and save for all datasets
for dataset_name, params in all_parameters.items():
    dataset_info = filtered_datasets[dataset_name]
    output_dir = dataset_info['output_dir']
    
    times_numeric = params['times_numeric']
    aromatic280_centers = params['aromatic280_centers']
    aromatic280_amps = params['aromatic280_amps']
    aromatic280_widths = params['aromatic280_widths']
    aromatic310_centers = params['aromatic310_centers']
    aromatic310_amps = params['aromatic310_amps']
    aromatic310_widths = params['aromatic310_widths']
    
    # Calculate amplitude ratio
    amplitude_ratio = [a280 / a310 if a310 > 0 else 0 for a280, a310 in zip(aromatic280_amps, aromatic310_amps)]
    
    # Create results dataframe
    results_df = pd.DataFrame({
        'time': times_numeric,
        'aromatic280_center': aromatic280_centers,
        'aromatic280_amplitude': aromatic280_amps,
        'aromatic280_width': aromatic280_widths,
        'aromatic310_center': aromatic310_centers,
        'aromatic310_amplitude': aromatic310_amps,
        'aromatic310_width': aromatic310_widths,
        'amplitude_ratio_280_310': amplitude_ratio
    })
    
    # Save to CSV
    csv_file = output_dir / f'{dataset_name}_results.csv'
    results_df.to_csv(csv_file, index=False)
    print(f"{dataset_name} - Results saved to {csv_file}")

## Export Detailed Fit Reports

In [ ]:
# Write detailed fit reports for all datasets
for dataset_name, fit_results in all_fit_results.items():
    dataset_info = filtered_datasets[dataset_name]
    output_dir = dataset_info['output_dir']
    data_filtered = dataset_info['data_filtered']
    
    report_file = output_dir / f'{dataset_name}_fit_reports.log'
    with open(report_file, 'w') as f:
        for i, (time_point, fit) in enumerate(zip(data_filtered.columns, fit_results)):
            if fit is not None:
                f.write(f'\n{"="*80}\n')
                f.write(f'Time point: {time_point}\n')
                f.write(f'{"="*80}\n')
                f.write(fit.fit_report(min_correl=0.3))
                f.write('\n')
    
    print(f"{dataset_name} - Detailed fit reports saved to {report_file}")

## Summary Statistics

In [ ]:
# Summary Statistics for all datasets
print("\n" + "="*80)
print("SPECTRAL ANALYSIS SUMMARY - UNLABELED SAMPLES")
print("="*80)

for dataset_name, params in all_parameters.items():
    aromatic280_centers = params['aromatic280_centers']
    aromatic280_amps = params['aromatic280_amps']
    aromatic310_centers = params['aromatic310_centers']
    aromatic310_amps = params['aromatic310_amps']
    
    # Calculate amplitude ratio
    amplitude_ratio = [a280 / a310 if a310 > 0 else 0 for a280, a310 in zip(aromatic280_amps, aromatic310_amps)]
    
    print(f"\n{dataset_name}:")
    print(f"  Aromatic 280nm Peak:")
    print(f"    Center range: {np.min(aromatic280_centers):.2f} - {np.max(aromatic280_centers):.2f} nm")
    print(f"    Mean center: {np.mean(aromatic280_centers):.2f} ± {np.std(aromatic280_centers):.2f} nm")
    print(f"    Amplitude range: {np.min(aromatic280_amps):.4f} - {np.max(aromatic280_amps):.4f}")
    print(f"    Mean amplitude: {np.mean(aromatic280_amps):.4f} ± {np.std(aromatic280_amps):.4f}")
    
    print(f"  Aromatic 310nm Peak:")
    print(f"    Center range: {np.min(aromatic310_centers):.2f} - {np.max(aromatic310_centers):.2f} nm")
    print(f"    Mean center: {np.mean(aromatic310_centers):.2f} ± {np.std(aromatic310_centers):.2f} nm")
    print(f"    Amplitude range: {np.min(aromatic310_amps):.4f} - {np.max(aromatic310_amps):.4f}")
    print(f"    Mean amplitude: {np.mean(aromatic310_amps):.4f} ± {np.std(aromatic310_amps):.4f}")
    
    print(f"  Amplitude Ratio (280/310):")
    print(f"    Range: {np.min(amplitude_ratio):.3f} - {np.max(amplitude_ratio):.3f}")
    print(f"    Mean: {np.mean(amplitude_ratio):.3f} ± {np.std(amplitude_ratio):.3f}")

print("\n" + "="*80)